# For viewing Images

**WARNING Execute "match_lables_with_gt.ipynb" beforehand!**

## Functions for constructing the images

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import pandas as pd
from utils import PROJECT_ROOT, SPLITS, IMG_SIZES, resolve_frame_paths

In [2]:
bounding_boxes = {}
for split in SPLITS:
    df = pd.read_csv(PROJECT_ROOT/f"data/bounding_boxes_with_labels/{split}.csv")
    df["frame_name"] = df["flight"].astype(str)+"_"+df["frame"].astype(str)
    bounding_boxes[split] = df
bounding_boxes["train"].head(5)

,flight,frame,class_id,species,txt_x1,txt_y1,txt_x2,txt_y2,txt_h,txt_w,frame_name
0,1,11726,1,Cervus elaphus (Red deer),709,578,749,632,53,40,1_11726
1,17,14339,5,Cervus elaphus (Red deer),456,102,510,146,43,54,17_14339
2,17,14342,5,Cervus elaphus (Red deer),454,115,507,161,46,52,17_14342
3,17,14343,5,Cervus elaphus (Red deer),449,141,506,191,50,56,17_14343
4,17,14349,5,Cervus elaphus (Red deer),447,178,504,228,49,57,17_14349


In [ ]:
def show_frame_with_boxes(frame_id: str, boxes: pd.DataFrame):
    """Load a frame by id, auto-detect split, and visualize YOLO boxes."""
    boxes = boxes[boxes["frame_name"] == frame_id]
    split, image_path, label_path = resolve_frame_paths(frame_id)

    image = Image.open(image_path).convert("RGB")
    img_w, img_h = image.size



    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(image)

    for class_id, x_c, y_c, w, h in boxes[]:
        x_min, y_min, box_w, box_h = yolo_to_xyxy(x_c, y_c, w, h, img_w, img_h)
        print(x_min, y_min, box_w, box_h)
        rect = patches.Rectangle((x_min, y_min), box_w, box_h, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x_min, max(y_min - 6, 2), f"class {class_id}", color="yellow", fontsize=10, backgroundcolor="black")

    ax.set_title(f"Frame {frame_id} from {split} with YOLO boxes")
    ax.axis("off")
    plt.show()

    print(f"Split: {split}")
    print(f"Image size: {img_w} x {img_h}")
    print(f"Label file: {label_path.name}")
    print(f"Number of boxes: {len(boxes)}")

    return {
        "split": split,
        "image_path": image_path,
        "label_path": label_path,
        "image_size": (img_w, img_h),
        "num_boxes": len(boxes),
        "boxes": boxes,
    }


# Demo: show the same frame currently used in this notebook.
frame_info = show_frame_with_boxes("1_11726")